# 04 — Advanced Tree & Boosting Models

Train and compare:
- Random Forest
- XGBoost
- CatBoost (native categorical features)

All models use a log-price target and the same leakage-safe train/test split.

In [1]:
from pathlib import Path
import sys, json, os

def _find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "bonbanh_usedcar_dataset.csv").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError("Project root not found. Keep notebooks inside the CarValue_AI_Complete_Project folder.")

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /mnt/data/CarValue_AI_Complete_Project


In [2]:
import time
import pandas as pd
import numpy as np
from src.carvalue_utils import (
    ensure_modeling_artifacts, get_train_test, TARGET, regression_metrics, save_bundle,
    fit_random_forest_bundle, fit_xgboost_bundle, fit_catboost_bundle
)

model_df, reference_year = ensure_modeling_artifacts(PROJECT_ROOT)
train, test = get_train_test(model_df)
y_test = test[TARGET].to_numpy(float)
print("Train/Test:", train.shape, test.shape)


Train/Test: (18533, 30) (4654, 30)


In [3]:
results = []
predictions = {"row_id": test.index, "actual_price_vnd": y_test}

def train_eval(name, fit_fn, filename):
    t0 = time.perf_counter()
    bundle = fit_fn(train, reference_year)
    seconds = time.perf_counter() - t0
    pred = bundle.predict(test)
    met = regression_metrics(y_test, pred)
    met["Train_seconds"] = seconds
    results.append({"Model": name, **met})
    predictions[name] = pred
    save_bundle(bundle, PROJECT_ROOT / "models" / filename)
    print(f"{name}: MAE={met['MAE_million_VND']:.2f}M | RMSE={met['RMSE_million_VND']:.2f}M | R2={met['R2']:.4f} | {seconds:.1f}s")
    return bundle


In [4]:
rf = train_eval("Random Forest", fit_random_forest_bundle, "random_forest.joblib")


Random Forest: MAE=82.32M | RMSE=322.04M | R2=0.9495 | 1.1s


In [5]:
xgb = train_eval("XGBoost", fit_xgboost_bundle, "xgboost.joblib")


XGBoost: MAE=85.70M | RMSE=299.26M | R2=0.9564 | 0.7s


In [6]:
cat = train_eval("CatBoost", fit_catboost_bundle, "catboost.joblib")


CatBoost: MAE=104.15M | RMSE=353.05M | R2=0.9393 | 6.1s


In [7]:
metrics_df = pd.DataFrame(results).sort_values("MAE_VND").reset_index(drop=True)
pred_df = pd.DataFrame(predictions)
metrics_df.to_csv(PROJECT_ROOT / "artifacts" / "metrics_advanced.csv", index=False)
pred_df.to_csv(PROJECT_ROOT / "artifacts" / "predictions_advanced.csv", index=False)
display(metrics_df.round(4))


,Model,MAE_VND,MAE_million_VND,RMSE_VND,RMSE_million_VND,R2,RMSLE,Train_seconds
0,Random Forest,8.232113e+07,82.3211,3.220409e+08,322.0409,0.9495,0.1420,1.1464
1,XGBoost,8.570358e+07,85.7036,2.992636e+08,299.2636,0.9564,0.1379,0.6669
2,CatBoost,1.041474e+08,104.1474,3.530539e+08,353.0539,0.9393,0.1402,6.0659
